# Myanmar ASR with Whisper Small

This notebook uses every source in the `notebook/dataset-mapping/*_whisper.csv` train, validation, and held-out test manifests. Audio paths are resolved under `dataset/`; synthetic `gemini/` clips remain excluded by the manifests. Read [README.md](README.md) before running it.

On Windows, follow the PowerShell setup in [README.md](README.md) before opening the notebook. Run cells top to bottom. `QUICK_RUN = False` uses all eligible clips by default. Set it to `True` for a short pipeline check. The three test manifests are evaluated only in full mode after training.


## 1. Install the notebook dependencies

First install PyTorch in this Jupyter kernel using the [official selector](https://pytorch.org/get-started/locally/) for your CPU/GPU. Restart the kernel after installing PyTorch. FFmpeg must also be on `PATH`. This cell installs the remaining packages; the notebook targets Transformers 4.57.1. Restart the kernel after this cell if Jupyter asks.


In [1]:
%pip install -q "transformers==4.57.1" "accelerate>=1.0" "soundfile>=0.12" "jiwer>=3" "tqdm>=4" "numpy>=1.24" "ffmpeg-python>=0.2" "huggingface_hub[hf_xet]"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Set paths and check hardware

Start Jupyter from anywhere inside the `myanmar-asr/` repository so the notebook can find its `dataset/` and `notebook/` folders. Full mode uses all eligible clips from every source. Set `QUICK_RUN = True` to select 256 train and 64 validation clips for a 20-step check.


In [2]:
import csv
import json
import random
import shutil
import subprocess
import unicodedata
from pathlib import Path
from collections import Counter

import jiwer
import numpy as np
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          WhisperForConditionalGeneration, WhisperProcessor)

cwd = Path.cwd().resolve()
ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "dataset").is_dir()
     and (p / "notebook" / "whisper-small").is_dir()),
    None,
)

print(ROOT)

d:\Development\myanmar-asr\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\Development\myanmar-asr


In [3]:
assert ROOT is not None, f"Start Jupyter from inside myanmar-asr/: {cwd}"

DATASET_DIR = ROOT / "dataset"
MANIFEST_DIR = ROOT / "notebook" / "dataset-mapping"
QUICK_RUN = False
MODEL_ID = "openai/whisper-small"
RUN_DIR = ROOT / "notebook" / "whisper-small" / ("run_all_quick" if QUICK_RUN else "run_all_full")
CACHE_DIR = ROOT / "notebook" / "whisper-small" / "cache"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(DATASET_DIR)
print(MANIFEST_DIR)

assert (MANIFEST_DIR / "train_whisper.csv").is_file()

D:\Development\myanmar-asr\dataset
D:\Development\myanmar-asr\notebook\dataset-mapping


In [4]:
# Check the device cuda

device = "cuda" if torch.cuda.is_available() else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
print({"root": str(ROOT), "device": device, "torch": torch.__version__,
       "transformers": transformers.__version__, "ffmpeg": shutil.which("ffmpeg")})

if device == "cpu":
    print("CPU can check the data pipeline, but full Whisper training needs a GPU in practice.")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("Current GPU:", torch.cuda.current_device())

elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Apple GPU (MPS) is available")

else:
    print("WARNING: Running on CPU")

{'root': 'D:\\Development\\myanmar-asr', 'device': 'cuda', 'torch': '2.11.0+cu128', 'transformers': '4.57.1', 'ffmpeg': 'C:\\Users\\admin\\AppData\\Local\\Microsoft\\WinGet\\Packages\\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\\ffmpeg-9.0.2-full_build\\bin\\ffmpeg.EXE'}
GPU: NVIDIA GeForce RTX 5060 Ti
GPU count: 1
Current GPU: 0


In [5]:
import shutil

print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))

ffmpeg: C:\Users\admin\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.2-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\admin\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-9.0.2-full_build\bin\ffprobe.EXE


## 3. Load every manifest split and check leakage

The manifests contain all supported speech sources and exclude clips longer than 30 seconds. This cell checks that each audio file exists and that training clips do not overlap evaluation clips by audio hash, record group, or transcript. The three held-out test sets are evaluated only after training.


In [6]:
TEST_SPLIT_NAMES = ("test_asr", "test_user_study", "test_ucsy")

def load_split(name):
    with (MANIFEST_DIR / f"{name}_whisper.csv").open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        required = {"sample_id", "audio", "source", "duration_s", "sha256", "text"}
        assert required <= set(reader.fieldnames or []), f"Missing columns in {name}"
        rows = list(reader)
    assert rows, f"No rows in {name}"
    for row in rows:
        path = (DATASET_DIR / row["audio"]).resolve()
        assert path.is_relative_to(DATASET_DIR.resolve()), row["audio"]
        assert path.is_file() and path.stat().st_size > 0, row["audio"]
        assert 0 < float(row["duration_s"]) <= 30, row["sample_id"]
    return rows

all_train = load_split("train")
all_validation = load_split("validation")
all_test_splits = {name: load_split(name) for name in TEST_SPLIT_NAMES}

def split_values(rows, key):
    if key == "record_group":
        return {(row["source"], row[key]) for row in rows if row.get(key)}
    return {row[key] for row in rows if row.get(key)}

for key in ("sha256", "record_group", "text"):
    training_values = split_values(all_train, key)
    for name, rows in {"validation": all_validation, **all_test_splits}.items():
        assert not training_values & split_values(rows, key), f"{key} crosses train and {name}"
for key in ("sha256", "record_group"):
    validation_values = split_values(all_validation, key)
    for name, rows in all_test_splits.items():
        assert not validation_values & split_values(rows, key), f"{key} crosses validation and {name}"

def summary(rows):
    return {"clips": len(rows), "hours": round(sum(float(r["duration_s"]) for r in rows) / 3600, 3),
            "sources": dict(sorted(Counter(r["source"] for r in rows).items()))}

print({name: summary(rows) for name, rows in
       {"train": all_train, "validation": all_validation, **all_test_splits}.items()})
rng = random.Random(20261030)
train_rows, validation_rows = all_train.copy(), all_validation.copy()
if QUICK_RUN:
    rng.shuffle(train_rows)
    rng.shuffle(validation_rows)
    train_rows = train_rows[:256]
    validation_rows = validation_rows[:64]
print("Selected:", summary(train_rows), summary(validation_rows))


{'train': {'clips': 16624, 'hours': 36.729, 'sources': {'asr': 7297, 'ucsy': 9327}}, 'validation': {'clips': 1390, 'hours': 2.4, 'sources': {'asr': 213, 'ucsy': 1177}}, 'test_asr': {'clips': 193, 'hours': 0.532, 'sources': {'asr': 193}}, 'test_user_study': {'clips': 153, 'hours': 0.504, 'sources': {'asr': 153}}, 'test_ucsy': {'clips': 1171, 'hours': 1.903, 'sources': {'ucsy': 1171}}}
Selected: {'clips': 16624, 'hours': 36.729, 'sources': {'asr': 7297, 'ucsy': 9327}} {'clips': 1390, 'hours': 2.4, 'sources': {'asr': 213, 'ucsy': 1177}}


## 4. Listen to two labels

These selected examples are a manual spot check, not an automatic proof that all labels are correct. Listen for text/audio mismatch, clipped speech, or a synthetic voice. If you find systematic problems, fix the manifest before trusting a model score.


In [7]:
for row in train_rows[:2]:
    print(row["source"], row["sample_id"], row["text"])
    display(Audio(filename=str(DATASET_DIR / row["audio"])))


asr asr:train:htluttaw-dawkhinsanhlaing_8046 လေးစားအပ် ပါ သော ပြည်ထောင်စုလွှတ်တော် နာယကကြီး နှင့်တကွ အားလုံး မင်္ဂလာ ပါ ရှင်


asr asr:train:htluttaw-dawkhinsanhlaing_8047 ဝန်ကြီး ရှင်းပြ တာ ကို မှတ်သား ထား ရပါတယ် ရှင်


## 5. Load Whisper and filter labels to its target length

The model downloads on first use. Myanmar transcription is selected explicitly. Rows whose target token sequence exceeds the model's limit are written to a review CSV, with counts printed here.


In [8]:
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="myanmar", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.language = "myanmar"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False  # Required when using gradient checkpointing.
model.generation_config.use_cache = True
MAX_LABEL_TOKENS = model.config.max_target_positions

def filter_label_length(rows):
    accepted, rejected = [], []
    for row in rows:
        length = len(processor.tokenizer(row["text"]).input_ids)
        (accepted if length < MAX_LABEL_TOKENS else rejected).append(row)
    return accepted, rejected

train_rows, rejected_train = filter_label_length(train_rows)
validation_rows, rejected_validation = filter_label_length(validation_rows)
assert train_rows and validation_rows
with (RUN_DIR / "long_labels.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["sample_id", "audio", "text"])
    writer.writeheader()
    writer.writerows({key: row[key] for key in writer.fieldnames}
                     for row in rejected_train + rejected_validation)
print({"max_label_tokens": MAX_LABEL_TOKENS, "train": summary(train_rows),
       "validation": summary(validation_rows), "long_labels": len(rejected_train) + len(rejected_validation)})


Token indices sequence length is longer than the specified maximum sequence length for this model (1092 > 1024). Running this sequence through the model will result in indexing errors


{'max_label_tokens': 448, 'train': {'clips': 12431, 'hours': 21.301, 'sources': {'asr': 3524, 'ucsy': 8907}}, 'validation': {'clips': 1247, 'hours': 1.922, 'sources': {'asr': 94, 'ucsy': 1153}}, 'long_labels': 4336}


## 6. Convert selected WAVs to 16 kHz mono

FFmpeg converts each selected input to a deterministic cache path. It runs once per distinct source hash; rerunning this cell reuses completed files. Keep several GB free for a full run. If conversion fails, inspect that file rather than silently dropping it.


In [9]:
def cache_audio(rows):
    for row in tqdm(rows, desc="Preparing 16 kHz audio"):
        target = CACHE_DIR / f"{row['sha256']}.wav"
        if not target.is_file():
            temporary = CACHE_DIR / f"{row['sha256']}.partial.wav"
            subprocess.run(["ffmpeg", "-nostdin", "-hide_banner", "-loglevel", "error", "-y",
                            "-i", str(DATASET_DIR / row["audio"]), "-ac", "1", "-ar", "16000",
                            "-c:a", "pcm_s16le", str(temporary)], check=True)
            temporary.replace(target)
        info = sf.info(target)
        assert info.samplerate == 16000 and info.channels == 1 and info.frames > 0, target
        row["cache_path"] = str(target)

cache_audio(train_rows + validation_rows)
print("Cache:", CACHE_DIR)


Preparing 16 kHz audio: 100%|██████████| 13678/13678 [00:34<00:00, 396.86it/s]

Cache: D:\Development\myanmar-asr\notebook\whisper-small\cache


## 7. Create the training dataset and scoring function

The loader computes Whisper features when a clip is requested. The collator pads text labels and masks padding from the loss. The primary score is CER after removing whitespace, which reduces the effect of inconsistent Myanmar word spacing; raw CER is also reported. Both scores keep punctuation.


In [10]:
class SpeechDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        audio, rate = sf.read(row["cache_path"], dtype="float32")
        assert rate == 16000 and audio.ndim == 1
        features = processor.feature_extractor(audio, sampling_rate=rate).input_features[0]
        labels = processor.tokenizer(row["text"]).input_ids
        return {"input_features": features, "labels": labels}

class SpeechCollator:
    def __call__(self, features):
        audio_inputs = [{"input_features": item["input_features"]} for item in features]
        batch = processor.feature_extractor.pad(audio_inputs, return_tensors="pt")
        text_inputs = [{"input_ids": item["labels"]} for item in features]
        labels = processor.tokenizer.pad(text_inputs, return_tensors="pt")
        label_ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
        if (label_ids[:, 0] == processor.tokenizer.bos_token_id).all().item():
            label_ids = label_ids[:, 1:]
        batch["labels"] = label_ids
        return batch

def no_spaces(text):
    return "".join(unicodedata.normalize("NFC", text).split())

def compute_metrics(result):
    predictions = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    labels = np.where(result.label_ids == -100, processor.tokenizer.pad_token_id, result.label_ids)
    hypotheses = processor.tokenizer.batch_decode(predictions, skip_special_tokens=True)
    references = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
    return {"cer_no_space": 100 * jiwer.cer([no_spaces(x) for x in references],
                                              [no_spaces(x) for x in hypotheses]),
            "cer_raw": 100 * jiwer.cer(references, hypotheses)}

train_dataset = SpeechDataset(train_rows)
validation_dataset = SpeechDataset(validation_rows)
sample = train_dataset[0]
assert sample["input_features"].ndim == 2 and sample["labels"]
assert SpeechCollator()([sample])["labels"].shape[0] == 1
print("One sample loaded and collated successfully.")


You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


One sample loaded and collated successfully.


## 8. Configure a short check or a full training run

The full run trains for three epochs using every eligible training clip and chooses the checkpoint with the lowest validation CER. Quick mode trains for 20 optimizer steps. CUDA uses mixed precision; other devices use full precision. Training uses one clip per GPU batch with 16 accumulation steps (effective batch size 16), and validation uses one clip per GPU batch to reduce peak memory. Resumable checkpoints are saved every 5 quick-run steps or 50 full-run steps, with up to three retained including the best. All-source runs have their own output folders, separate from earlier UCSY-only runs. If the display goes black during training, check the Windows System log for a GPU driver reset before retrying.


In [11]:
interval = 20 if QUICK_RUN else 500
checkpoint_interval = 5 if QUICK_RUN else 50

class FrequentCheckpointCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % checkpoint_interval == 0:
            control.should_save = True
        return control

training_args = Seq2SeqTrainingArguments(
    output_dir=str(RUN_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-5,
    warmup_steps=5 if QUICK_RUN else 100,
    num_train_epochs=3,
    max_steps=20 if QUICK_RUN else -1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    fp16=(device == "cuda"),
    optim="adamw_torch",
    eval_strategy="steps",
    eval_steps=interval,
    save_strategy="steps",
    save_steps=interval,
    save_total_limit=3,
    logging_steps=5 if QUICK_RUN else 50,
    predict_with_generate=True,
    generation_max_length=MAX_LABEL_TOKENS,
    eval_accumulation_steps=4,
    dataloader_num_workers=0,  # Keeps notebook data loading reliable on Windows.
    load_best_model_at_end=True,
    metric_for_best_model="cer_no_space",
    greater_is_better=False,
    report_to=[],
    push_to_hub=False,
    seed=20261030,
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=SpeechCollator(),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[FrequentCheckpointCallback()],
)
print("Output:", RUN_DIR, "| checkpoint every", checkpoint_interval, "steps")


Output: D:\Development\myanmar-asr\notebook\whisper-small\run_all_full | checkpoint every 50 steps


## 9. Measure the pretrained baseline, then fine-tune

The baseline uses the exact same validation clips and scoring function as training. After a power cut, restart the kernel and rerun cells from the top; this cell reuses saved baseline metrics and resumes from the newest complete checkpoint. An interrupted checkpoint is skipped. After fine-tuning, the best checkpoint is saved with the processor. Compare the printed metrics before deciding whether the model improved.


In [12]:
import zipfile

def latest_complete_checkpoint():
    checkpoints = sorted(
        (p for p in RUN_DIR.glob("checkpoint-*") if p.is_dir() and p.name[11:].isdigit()),
        key=lambda p: int(p.name[11:]), reverse=True,
    )
    required = ("trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth")
    for checkpoint in checkpoints:
        try:
            if not all((checkpoint / name).stat().st_size > 0 for name in required):
                raise ValueError("missing or empty resume state")
            if not any((checkpoint / name).stat().st_size > 0 for name in
                       ("model.safetensors", "pytorch_model.bin") if (checkpoint / name).is_file()):
                raise ValueError("missing or empty model weights")
            # A nonempty file can still be a truncated PyTorch ZIP archive.
            archives = [checkpoint / name for name in required[1:]]
            archives += [checkpoint / name for name in ("scaler.pt", "pytorch_model.bin")
                         if (checkpoint / name).is_file()]
            for path in archives:
                try:
                    with zipfile.ZipFile(path) as archive:
                        bad_member = archive.testzip()
                        if bad_member is not None:
                            raise ValueError(f"CRC check failed: {bad_member}")
                except (OSError, ValueError, zipfile.BadZipFile, EOFError, RuntimeError) as error:
                    raise ValueError(f"{path.name}: {error}") from error
            state = json.loads((checkpoint / "trainer_state.json").read_text(encoding="utf-8"))
            if state.get("global_step") != int(checkpoint.name[11:]):
                raise ValueError("trainer step does not match checkpoint folder")
            return checkpoint
        except (OSError, ValueError) as error:
            print(f"Skipping {checkpoint.name}: {error}")
    if checkpoints:
        raise RuntimeError(
            "No checkpoint passed the resume checks. Existing checkpoints were preserved. "
            "Restore a backup or use a new RUN_DIR to start over."
        )
    return None

checkpoint = latest_complete_checkpoint()
baseline_path = RUN_DIR / "baseline_metrics.json"
try:
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    print("Pretrained baseline (saved):", baseline)
except (FileNotFoundError, json.JSONDecodeError):
    baseline = trainer.evaluate(metric_key_prefix="baseline")
    print("Pretrained baseline:", baseline)
    baseline_path.write_text(json.dumps(baseline, indent=2), encoding="utf-8")
print("Resuming from:" if checkpoint else "Starting new training in:", checkpoint or RUN_DIR)

train_result = trainer.train(resume_from_checkpoint=str(checkpoint) if checkpoint else None)

print("Training:", train_result.metrics)

best_dir = RUN_DIR / "best_model"
trainer.save_model(str(best_dir))
processor.save_pretrained(str(best_dir))

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Validation after training:", trainer.evaluate(metric_key_prefix="validation"))


Pretrained baseline (saved): {'baseline_loss': 2.4502601623535156, 'baseline_model_preparation_time': 0.0033, 'baseline_cer_no_space': 263.7910424985376, 'baseline_cer_raw': 294.7870575221239, 'baseline_runtime': 4571.4278, 'baseline_samples_per_second': 0.273, 'baseline_steps_per_second': 0.273}
Resuming from: D:\Development\myanmar-asr\notebook\whisper-small\run_all_full\checkpoint-1700


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.

In [ ]:
row = validation_rows[0]
audio, rate = sf.read(row["cache_path"], dtype="float32")
inputs = processor(audio, sampling_rate=rate, return_tensors="pt")
inputs = {key: value.to(trainer.model.device) for key, value in inputs.items()}
with torch.no_grad():
    output_ids = trainer.model.generate(**inputs, max_new_tokens=256)
print("Reference:", row["text"])
print("Prediction:", processor.batch_decode(output_ids, skip_special_tokens=True)[0])
